# Data Management - Exisiting Values

**Learning Objective:** 
- Learn to recode and create variables
- Learn about the `.replace()` method and the `pd.cut()` function
- Learn to compare variables


In [ ]:
# Load Pandas
import pandas as pd

# Import Data
data_url = "https://raw.githubusercontent.com/datamisc/ts-2020/main/data.csv"
anes_data  = pd.read_csv(data_url, compression='gzip')


In [ ]:
# Subsetting & Renaming Variables
my_vars = [
    "V201033",  # vote_int
    "V201507x",  # age
    "V201200",  # liberal-conservative self-placement
    "V201151",	 # biden thrm
    "V201152",  # trump thrm
    "V201144x",  # covid 
    "V201233",  # trust_gov
    "V201236",  # trust_corrupt
    "V201237",  # trust_people
]

df = anes_data[my_vars]

df.columns = ["vote_int", "age", "ideology", "biden_thrm", "trump_thrm", "covid", "trust_gov", "trust_corrupt", "trust_people"]
df.head()


## Recoding Variables Using a Mask/Filter

We can use relational operators (<>=) to identify the observations that meet certain criteria that we want to change.

For instance we could recode the voting intention variable:

- [V201033](https://sda.berkeley.edu/sdaweb/docs/nes2020/DOC/hcbk0003.htm#V201033)



In [ ]:
# Recoding values with a mask
new_df = df[(df['vote_int']>0) & (df['vote_int']<=5)].copy()  # Creating a new dataframe

In [ ]:
new_df

In [ ]:
# Recoding for value == 1
mask = new_df['vote_int'] == 1
new_df.loc[mask, "vote_int"] = "J.Biden"

In [ ]:
new_df

In [ ]:
mask = new_df['vote_int'] == 2
new_df.loc[mask, "vote_int"] = "D.Trump"


In [ ]:
new_df['vote_int'].value_counts()


### Hack-Time



In [ ]:
# Finish recoding the `vote_int` variable


## Recoding Variables Using `replace()`

Recoding each category one by one is tedious.

To make this easier, one can also recode variables using the `replace()` method.

Let's recode the `covid` variable this time using the `.replace()` method and dictionary!

- [V201144x](https://sda.berkeley.edu/sdaweb/docs/nes2020/DOC/hcbk0005.htm#V201144x)

In [ ]:
# What does the covid variable look like?
new_df['covid'].value_counts()

In [ ]:
# Creating two lists with the old and the new labels
old_labels = [-2, 1, 2, 3, 4]
new_labels = ["Don't Know", "1.Approve strongly", "2.Approve not strongly", "3.Disapprove not strongly", "4.Disapprove strongly"]


In [ ]:
new_df['covid'].replace(old_labels, new_labels)


In [ ]:
# This looks good!
new_df['covid'].replace(old_labels, new_labels).value_counts()

In [ ]:
# Once we're happy with the result, we can save/assign it to our previous variable
new_df['covid'] = new_df['covid'].replace(old_labels, new_labels)
# And we take a look again at the variable
new_df['covid'].value_counts(normalize=True)

In [ ]:
new_df['covid'].value_counts(normalize=True).sort_index().plot(kind='bar')


### Hack-Time

In [ ]:
# Are citizens who approve the actions taken by the government to 
# handle covid more likely to liberal or conservative?
# TIP: use pd.crosstabs!


## Filtering or Recoding?

Until now, we have been filtering out observations that we don't need without thinking about the consequences they can have on our results. 

If you apply multiple filters, you add bias to your dataset but you might also end up loosing a big part of your data! 
- **With less data we have less evidence to draw conclusions!**

Let's try to predict the election outcome using pre-election data!

In [ ]:
# Filtering Out ALL Observations that are note Biden or Trump
mask = df['vote_int'].between(1,2)
filter_df = df[mask].copy()

# Change Labels
vote_int_labels = {
    1: "J.Biden",
    2: "D.Trump",
}

filtered_output = filter_df['vote_int'].replace(vote_int_labels).value_counts(normalize=True)
filtered_output

In [ ]:
# Recoding Observations
mask = df["vote_int"] > 0
recoded_df = df[mask].copy()

# Keeping all other vote choice intentions coded as "Other"
mask = recoded_df["vote_int"] > 2
recoded_df.loc[mask, "vote_int"] = "Other"

# Recoding the remaining labels using the previously created dictionary
recoded_df["vote_int"] = recoded_df["vote_int"].replace(vote_int_labels)

recoded_output = recoded_df["vote_int"].value_counts(normalize=True)
recoded_output


In [ ]:
print("The Filtered Output")
print(filtered_output)
print("==============================")
print("The Recoded Output")
print(recoded_output)

# Creating New Variables (~ Adding New Columns)

When you recode variables you might want to add a new variable to the original dataset to keep the orignial version of your variable.

![](https://pandas.pydata.org/docs/_images/05_newcolumn_1.svg)




In [ ]:
recoded_df

In [ ]:
recoded_df['my_new_var'] = 0
recoded_df

In [ ]:
# We can also remove a column using the drop method
# Let's look at how the recoded_df if we remove/drop the variable
recoded_df.drop('my_new_var', axis=1)

In [ ]:
# Once we're happy with the output, we save!
recoded_df = recoded_df.drop('my_new_var', axis=1)

## Hack-Time

In [ ]:
# Add a new binary variable that takes the value 1 when
# the respondent intends to vote for Trump. 
# Name this variable `vote_trump`


Let's now add a categorical age variable to our dataset!


In [ ]:
# But how?
mask = recoded_df['age'] >= 18
recoded_df['age_cat'] = "18-35"
mask = recoded_df['age'] >= 36 
recoded_df['age_cat'] = "36-50"
mask = recoded_df['age'] >= 51
recoded_df['age_cat'] = "51-65"
...
...
...

Once again this is very tedious...

## The cut function
The `pd.cut()` function allows us to convert a continuous variable into a discrete variable !

In [ ]:
recoded_df["age_cat"] = pd.cut(df["age"], bins=[17,35,50,65,80], labels=["18-35", "36-50", "51-65", "66+"])
recoded_df


In [ ]:
recoded_df['age_cat'].value_counts().sort_index().plot(kind='bar')


In [ ]:
pd.crosstab(recoded_df['vote_int'], recoded_df['age_cat'], normalize=True).plot(kind='bar', subplots=True, figsize=(10,10), layout=(2,2));

### Hack-Time

In [ ]:
# Add a cleaned version of the covid variable named `clean_covid` 


In [ ]:
# Which age group agrees more with how the governement is handling covid?


# Creating an Additive Scale ?
Additive scales combine multiple related survey items into a single measure by summing responses across rows.

## Building an Political Trust Additive Scale

In [ ]:
# Select three related survey items
trust_scale_vars = [ "trust_gov", "trust_corrupt", "trust_people" ]
df[trust_scale_vars]

In [ ]:
# Recode special missing values 
old_labels = [-1, -2, -3, -4, -5, -6, -7, -8, -9]
df[trust_scale_vars] = df[trust_scale_vars].replace(old_labels, pd.NA)

In [ ]:
# Reverse-code variables so higher values = more trust
# You need to check the codebook to understand the direction
df['trust_gov'] = 6 - df['trust_gov']
df['trust_people'] = 6 - df['trust_people']

# Did you notice we did not reverse the `trust_corrupt`/V201236 variable. Why?

In [ ]:
# Create additive scale by summing across items
df['trust_scale'] = df[trust_scale_vars].sum(axis=1)


In [ ]:
# Let's check if everything looks ok
print(df['trust_scale'].describe())
df['trust_scale'].value_counts().sort_index().plot(kind='bar')